# video-to-3d — phone video → 3D point cloud (Colab)

Modern, GPU path using **DUSt3R**. No COLMAP install, no camera calibration.

**Before you run:**
1. `Runtime → Change runtime type → Hardware accelerator: **T4 GPU**`
2. Edit `REPO_URL` in the setup cell to point at *your* fork.
3. `Runtime → Run all`, then upload a short clip when prompted.

Pipeline: **upload video → extract frames → blur filter → DUSt3R → view → save .ply**

## 0. Check the GPU
If this prints nothing, you forgot to switch the runtime to a T4 GPU.

In [ ]:
!nvidia-smi -L

## 1. Setup — clone the repo + DUSt3R, install deps
Edit `REPO_URL` to your own GitHub fork (that's how Colab gets the `v3d/` code).

In [ ]:
REPO_URL = "https://github.com/maheswariridhi/video-to-3d.git"  # <-- change if your repo differs

import os
if not os.path.isdir("video-to-3d"):
    !git clone $REPO_URL
%cd video-to-3d
!pip install -q -r requirements.txt

# DUSt3R — the reconstruction engine (cloned, not pip-installed).
if not os.path.isdir("dust3r"):
    !git clone --recursive https://github.com/naver/dust3r
    !pip install -q -r dust3r/requirements.txt
import sys
sys.path.append("dust3r")

## 2. Upload your video
A 10–30 s slow sweep of a small room works best. (Or mount Google Drive instead.)

In [ ]:
from google.colab import files
uploaded = files.upload()
video_path = list(uploaded.keys())[0]
print("uploaded:", video_path)

## 3. Prep frames (CPU): extract → deblur → thin to fit GPU memory

In [ ]:
from v3d import frames

frames.extract(video_path, "out/images", fps=2)
frames.filter_blurry("out/images")
frames.subsample("out/images", max_frames=25)  # denser coverage; stays on DUSt3R's "complete" graph (<=25)

## 4. Reconstruct (GPU): DUSt3R → coloured point cloud

In [ ]:
from v3d import reconstruct

rec = reconstruct.run("out/images", device="cuda")

## 5. (Optional) Semantic labels — roadmap
Not wired up yet. The plan is Grounded-SAM-2 per frame, then lift the 2D masks
onto `rec.frames[i].pts3d` and majority-vote per 3D point. See `v3d/semantics.py`.

## 6. View inline + save outputs
Renders the cloud here, then writes `point_cloud.ply`, a standalone
`preview.html`, and `report.json`, and downloads them all as one zip.

In [ ]:
import json
import shutil
from pathlib import Path

from google.colab import files
from v3d import pointcloud

# Interactive view inline + a standalone HTML a reviewer can open without Colab.
fig = pointcloud.show(rec.points, rec.colors)
fig.write_html("out/preview.html")

pointcloud.save_ply(rec.points, rec.colors, "out/point_cloud.ply")

report = {
    "num_frames_used": len(list(Path("out/images").glob("frame_*.jpg"))),
    "num_points": int(len(rec.points)),
    "output_ply": "out/point_cloud.ply",
    "preview_html": "out/preview.html",
}
Path("out/report.json").write_text(json.dumps(report, indent=2))
print(json.dumps(report, indent=2))

# Bundle everything (ply + preview.html + report.json + images/) into one download.
shutil.make_archive("video_to_3d_outputs", "zip", "out")
files.download("video_to_3d_outputs.zip")